# EMNIST Recogniser — Training Notebook

Trains `EMNISTNet`, a CNN with residual blocks, on the **EMNIST Balanced** dataset (47 classes: digits 0–9, uppercase A–Z, lowercase a–z).

**Runtime required:** GPU (T4 recommended)
`Runtime → Change runtime type → T4 GPU → Save`

**Expected training time:** ~25–35 minutes for 30 epochs on a T4
**Target accuracy:** ≥ 88% test accuracy (typically achieves ~90%)

---

### Before you start

You'll need to upload two files from your local `backend/` folder when prompted in Cell 2:
- `model.py`
- `train.py`


## 1 — Verify GPU & Environment

In [ ]:
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "⚠️  No GPU detected — switch runtime to GPU!")

import torch
import torchvision

print(f"PyTorch      : {torch.__version__}")
print(f"Torchvision  : {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2 — Upload `model.py` and `train.py`

In [ ]:
from google.colab import files as colab_files

print("Upload backend/model.py and backend/train.py from your local machine.")
print("Select BOTH files when the dialog opens.\n")
uploaded = colab_files.upload()

for fname in uploaded:
    print(f"  ✓ Uploaded: {fname} ({len(uploaded[fname]):,} bytes)")

assert "model.py" in uploaded, "model.py not found — please upload it!"
assert "train.py" in uploaded, "train.py not found — please upload it!"
print("\nReady to train. Proceed to Cell 3.")


## 3 — Train the Model

This downloads EMNIST Balanced (~560 MB, cached after first run), then trains for 30 epochs with OneCycleLR scheduling, label smoothing, and data augmentation.

The script saves `emnist_net.pth` automatically whenever validation accuracy improves — so the file you end up with corresponds to the **best epoch**, not necessarily the final one.


In [ ]:
!python train.py --epochs 30 --batch-size 256 --lr 1e-3 --output-dir .


## 4 — Sanity Check: Load Model & Run Sample Predictions

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import datasets, transforms

from model import EMNISTNet, EMNIST_LABELS, NUM_CLASSES

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = EMNISTNet(num_classes=NUM_CLASSES)
state = torch.load("emnist_net.pth", map_location=device)
model.load_state_dict(state)
model.to(device)
model.eval()

print("✓ Model loaded successfully")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
# Run on a handful of real test samples
test_ds = datasets.EMNIST(
    root="./data",
    split="balanced",
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.Lambda(lambda img: img.rotate(90)),
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])
)

print("Sample predictions:\n")
for i in range(5):
    img, label_idx = test_ds[i * 200]
    with torch.no_grad():
        logits = model(img.unsqueeze(0).to(device))
        probs  = F.softmax(logits, dim=1)[0]
        pred   = probs.argmax().item()
        conf   = probs[pred].item() * 100
    true_label = EMNIST_LABELS[label_idx]
    pred_label = EMNIST_LABELS[pred]
    status = "✓" if pred == label_idx else "✗"
    print(f"  {status} True: {true_label:>2} | Pred: {pred_label:>2} ({conf:.1f}%)")


## 5 — Full Test-Set Accuracy

Re-confirms the headline accuracy figure across the entire 18,800-sample test split (not just the 5 samples above).


In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=2)

correct, total = 0, 0
with torch.no_grad():
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        preds = model(imgs).argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"Full test-set accuracy: {correct:,}/{total:,} = {correct/total*100:.2f}%")


## 6 — Download the Trained Weights

> **Note:** Run this cell directly — do not call `files.download()` from inside a `!python script.py` subprocess. That fails with `AttributeError: 'NoneType' object has no attribute 'kernel'` because the subprocess has no connection to the Colab frontend's comm channel. Running it here, in a genuine notebook cell, works correctly.


In [ ]:
import os

assert os.path.exists("emnist_net.pth"), "Model file not found — did training complete?"
size_mb = os.path.getsize("emnist_net.pth") / 1024 / 1024
print(f"Model size: {size_mb:.1f} MB")
print("Downloading emnist_net.pth to your computer…")

colab_files.download("emnist_net.pth")

print("\n✓ Done! Place emnist_net.pth in the backend/ folder before running the API.")


## 7 — (Optional) Resume Training From a Checkpoint

If your Colab session disconnects mid-training, `train.py` saves a resumable checkpoint to `emnist_checkpoint.pth` after every epoch. Run the cell below to pick up where you left off rather than starting from scratch.


In [ ]:
# !python train.py --resume --epochs 30 --batch-size 256 --lr 1e-3 --output-dir .


---

### What You Should Have Now

- `emnist_net.pth` downloaded to your local machine
- Confirmation of test accuracy (typically ~90%)
- Sample predictions showing the model works on real handwritten characters

### Next Step

Move `emnist_net.pth` into your project's `backend/` folder, then start the FastAPI server:

```bash
cd backend
uvicorn main:app --reload
```

Look for `Model loaded in XX.X ms` in the startup log — that confirms the weights loaded correctly.
